1.Cadre légal et éthique

Une API existe-t-elle ?	Non
Que dit robots.txt ?	Fichier vide, aucune restriction
Que disent les CGU ?	Aucune restriction particulière constatée
Y a-t-il des données personnelles ?	Non
Combien de requêtes ?	Une catégorie uniquement, avec temporisation
Les résultats seront-ils conservés ?	Oui, dans un fichier CSV

2.Récupérer la page : requests

In [30]:
import requests

URL = "https://books.toscrape.com/catalogue/page-1.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (ton.email@exemple.fr)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"            # <- a ne pas oublier, on voit pourquoi juste apres

print("status   :", r.status_code)
print("longueur :", len(r.text), "caracteres")
print("debut    :", r.text[:60])

status   : 200
longueur : 50449 caracteres
debut    : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [31]:
r = requests.get(URL, headers=ENTETES, timeout=10)

print("encodage devine par requests :", r.encoding)
print("encodage reel de la page     :", r.apparent_encoding)
print("le prix :", r.text[r.text.find("price_color") : r.text.find("price_color") + 40])

encodage devine par requests : ISO-8859-1
encodage reel de la page     : utf-8
le prix : price_color">Â£51.77</p>
    

<p class=


In [32]:
r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

In [33]:
print("le prix est present :", "£51.77" in r.text)
print("le titre est present:", "A Light in the Attic" in r.text)

le prix est present : True
le titre est present: True


3.Appliquer tes sélecteurs: BeautifulSoup

In [34]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

fiches = soup.select("article.product_pod")
print("fiches trouvees :", len(fiches))

fiches trouvees : 20


Lire une donnée:contenu ou attribut

In [35]:
fiche = fiches[0]

# le CONTENU d'une balise
prix = fiche.select_one("p.price_color").get_text(strip=True)

# un ATTRIBUT
lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]
href = lien_h3["href"]

print("prix  :", prix)
print("titre :", titre)
print("href  :", href)

prix  : £51.77
titre : A Light in the Attic
href  : a-light-in-the-attic_1000/index.html


Lire une classe

In [36]:
classes = fiche.select_one("p.star-rating")["class"]
print("classes :", classes)

classes : ['star-rating', 'Three']


In [37]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

note = NOTES[classes[1]]
print("note :", note)

note : 3


Le lien relatif: urljoin

In [38]:
from urllib.parse import urljoin

print(urljoin(URL, href))

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


Tout assembler: une fonction par fiche

In [39]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }


print(lire_fiche(fiches[0], URL))

{'titre': 'A Light in the Attic', 'prix': '£51.77', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


4.Plusieurs pages : suivre la pagination

In [40]:
suivant = soup.select_one("li.next a")
print(suivant["href"] if suivant else "pas de page suivante")

page-2.html


In [41]:
import time

def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres = []
    page = 0

    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"  page {page} : status {r.status_code}, on s'arrete")
            break

        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"  page {page} : {len(fiches)} livres  (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)

    return livres


livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/page-1.html")
print("collecte :", len(livres), "livres")

  page 1 : 20 livres  (total 20)
  page 2 : 20 livres  (total 40)
  page 3 : 20 livres  (total 60)
collecte : 60 livres


5.Le second niveau: suivre les liens

In [42]:
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None}

    soup = BeautifulSoup(r.text, "html.parser")

    # le tableau : une entree par ligne, le th en cle et le td en valeur
    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    # la categorie : troisieme element du fil d'Ariane
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }


url_test = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
print(lire_detail(url_test))

{'categorie': 'Poetry', 'stock': 'In stock (22 available)', 'upc': 'a897fe39b1053632'}


Extraire le nombre du stock

In [43]:
import re

def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0


print(nombre_en_stock("In stock (22 available)"))
print(nombre_en_stock("Out of stock"))
print(nombre_en_stock(None))

22
0
None


Challenge - Collecter un catalogue sur deux niveaux
Étape 1 - reprendre ta spécification
Catégorie : Music

Nombre de livres annoncé : 13

Nombre de pages : 1

Bloc répété
Sélecteur : article.product_pod

Résultat console : 13

Tableau des données
Donnée	Sélecteur	Contenu ou attribut ?
titre	h3 a	attribut title
prix	p.price_color	contenu
disponibilité	p.instock.availability	contenu
note	p.star-rating	attribut class
lien vers la fiche	h3 a	attribut href
Deux sélecteurs testés
Sélecteur	Résultat
document.querySelectorAll('article.product_pod p.price_color').length	13
document.querySelectorAll('article.product_pod h3 a').length	13
Pagination
Aucun lien next.

La catégorie tient sur une seule page.

Contrôle du code source
Le prix a été trouvé dans le code source HTML.

Un script peut donc récupérer cette information directement avec requests.

Informations présentes sur la fiche détaillée
UPC : tableau Product Information
Number of reviews : tableau Product Information

étape 2-Une page,pour valider la chaîne.

In [45]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://books.toscrape.com/catalogue/category/books/music_14/index.html"
entetes = {"User-Agent": "bootcamp-data/1.0"}
r = requests.get(url, headers=entetes, timeout=10) 
r.encoding ="utf-8"
soup = BeautifulSoup(r.text, "html.parser")


In [46]:
fiches = soup.select("article.product_pod")
print("Nombre de fiches :", len(fiches))

Nombre de fiches : 13


In [47]:
NOTES = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

def lire_fiche(fiche, url_page):

    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]

    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"])
    }

In [48]:
livres = [lire_fiche(fiche, URL) for fiche in fiches]

df = pd.DataFrame(livres)

df

,titre,prix,dispo,note,url
0,Rip it Up and Start Again,£35.02,In stock,5,https://books.toscrape.com/rip-it-up-and-start...
1,Our Band Could Be Your Life: Scenes from the A...,£57.25,In stock,3,https://books.toscrape.com/our-band-could-be-y...
2,How Music Works,£37.32,In stock,2,https://books.toscrape.com/how-music-works_979...
3,Love Is a Mix Tape (Music #1),£18.03,In stock,1,https://books.toscrape.com/love-is-a-mix-tape-...
4,Please Kill Me: The Uncensored Oral History of...,£31.19,In stock,4,https://books.toscrape.com/please-kill-me-the-...
5,Kill 'Em and Leave: Searching for James Brown ...,£45.05,In stock,5,https://books.toscrape.com/kill-em-and-leave-s...
6,"Chronicles, Vol. 1",£52.60,In stock,2,https://books.toscrape.com/chronicles-vol-1_46...
7,This Is Your Brain on Music: The Science of a ...,£38.40,In stock,1,https://books.toscrape.com/this-is-your-brain-...
8,Orchestra of Exiles: The Story of Bronislaw Hu...,£12.36,In stock,3,https://books.toscrape.com/orchestra-of-exiles...
9,No One Here Gets Out Alive,£20.02,In stock,5,https://books.toscrape.com/no-one-here-gets-ou...


In [49]:
print(df["note"].unique())

[5 3 2 1 4]


In [50]:
print(df["titre"])

0                             Rip it Up and Start Again
1     Our Band Could Be Your Life: Scenes from the A...
2                                       How Music Works
3                         Love Is a Mix Tape (Music #1)
4     Please Kill Me: The Uncensored Oral History of...
5     Kill 'Em and Leave: Searching for James Brown ...
6                                    Chronicles, Vol. 1
7     This Is Your Brain on Music: The Science of a ...
8     Orchestra of Exiles: The Story of Bronislaw Hu...
9                            No One Here Gets Out Alive
10                                                 Life
11    Old Records Never Die: One Man's Quest for His...
12                     Forever Rockers (The Rocker #12)
Name: titre, dtype: str


étape 3- Toutes les pages de liste

In [51]:
import time
def collecter_toutes_pages(url_depart):
    url = url_depart
    livres = []
    while url:
        r =requests.get(url,headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres.extend([lire_fiche(fiche, url) for fiche in fiches])
        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None
        time.sleep(1)  # Respect the website's server
    return livres

In [53]:
livres = collecter_toutes_pages(url)
print("Nombre de livres:",len(livres))

Nombre de livres: 13


étape 4-Le second niveau

In [54]:
for i, livre in enumerate(livres, start=1):

    detail = lire_detail(livre["url"])

    livre["categorie"] = detail["categorie"]
    livre["stock"] = nombre_en_stock(detail["stock"])
    livre["upc"] = detail["upc"]

    print(f"Livre {i}/{len(livres)} traité")

    time.sleep(1)

Livre 1/13 traité
Livre 2/13 traité
Livre 3/13 traité
Livre 4/13 traité
Livre 5/13 traité
Livre 6/13 traité
Livre 7/13 traité
Livre 8/13 traité
Livre 9/13 traité
Livre 10/13 traité
Livre 11/13 traité
Livre 12/13 traité
Livre 13/13 traité


étape 5-Assembler et nettoyer

In [55]:
df = pd.DataFrame(livres)
df.head()


,titre,prix,dispo,note,url,categorie,stock,upc
0,Rip it Up and Start Again,£35.02,In stock,5,https://books.toscrape.com/catalogue/rip-it-up...,Music,19,a34ba96d4081e6a4
1,Our Band Could Be Your Life: Scenes from the A...,£57.25,In stock,3,https://books.toscrape.com/catalogue/our-band-...,Music,19,deda3e61b9514b83
2,How Music Works,£37.32,In stock,2,https://books.toscrape.com/catalogue/how-music...,Music,19,327f68a59745c102
3,Love Is a Mix Tape (Music #1),£18.03,In stock,1,https://books.toscrape.com/catalogue/love-is-a...,Music,14,adb5f51c2e2ab13f
4,Please Kill Me: The Uncensored Oral History of...,£31.19,In stock,4,https://books.toscrape.com/catalogue/please-ki...,Music,8,4a823d80aa30dbb0


In [56]:
df["prix_gbp"] = (
    df["prix"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

In [57]:
df = df[
    [
        "titre",
        "categorie",
        "prix_gbp",
        "note",
        "stock",
        "upc",
        "url"
    ]
]

df.head()

,titre,categorie,prix_gbp,note,stock,upc,url
0,Rip it Up and Start Again,Music,35.02,5,19,a34ba96d4081e6a4,https://books.toscrape.com/catalogue/rip-it-up...
1,Our Band Could Be Your Life: Scenes from the A...,Music,57.25,3,19,deda3e61b9514b83,https://books.toscrape.com/catalogue/our-band-...
2,How Music Works,Music,37.32,2,19,327f68a59745c102,https://books.toscrape.com/catalogue/how-music...
3,Love Is a Mix Tape (Music #1),Music,18.03,1,14,adb5f51c2e2ab13f,https://books.toscrape.com/catalogue/love-is-a...
4,Please Kill Me: The Uncensored Oral History of...,Music,31.19,4,8,4a823d80aa30dbb0,https://books.toscrape.com/catalogue/please-ki...


étape 6-Analyser

1.Combien de livre et quel stock total ?

In [58]:
print("nombre de livres :", len(df))
print("stock total:",df["stock"].sum())

nombre de livres : 13
stock total: 111


2.Quel est le prix moyen ?, et livre le plus cher ?

In [59]:
print("Prix moyen:",df["prix_gbp"].mean())

Prix moyen: 35.63692307692308


In [62]:
plus_cher = df.loc[df["prix_gbp"].idxmax()]
print("Livre le plus cher:", plus_cher["titre"])

Livre le plus cher: Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991


3.Comment ce répartissent les notes ?

In [64]:
repartition = df["note"].value_counts().sort_index()
print("répartition des notes:",repartition)

répartition des notes: note
1    2
2    3
3    3
4    1
5    4
Name: count, dtype: int64


4.la colone categorie contient-elle une valeur?

In [65]:
print(df["categorie"].unique())

<StringArray>
['Music']
Length: 1, dtype: str


La colone categorie contient une seule valeur : Music .
Cependant c'est normal car nous avions choisie une seul categorie au départ "Music".

In [66]:
df.to_csv("../Data/music.csv", index=False)